## Without web search

In [1]:
%run langchain_common.py

C:\Users\akhawaja\git\cs4603\wk3_langchain_agents\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [21]:
agent = create_agent(
    model=llm_noreason
)

In [3]:
agent.invoke(
    {"messages": [HumanMessage(content="who is current mayor of lahore?")]}
)['messages'][-1].content

'The current Mayor of Lahore is **Khalid Maqbool Siddiqui**.\n\nHe assumed office on **August 2, 2022**, representing the Pakistan Muslim League-Nawaz (PML-N). He was elected following the local government elections held in July 2022, succeeding the caretaker mayor, Chaudhry Muhammad Aslam.'

In [22]:
agent.invoke(
    {"messages": [HumanMessage(content="what is the weather today in lahore?")]}
)['messages'][-1].content

'I do not have access to real-time data, so I cannot tell you the current weather in Lahore today.\n\nTo get the most accurate and up-to-date forecast, I recommend:\n*   Searching **"Lahore weather"** on Google or Bing.\n*   Checking a dedicated weather website like **AccuWeather**, **Weather.com**, or **Windy**.\n*   Looking at the **Pakistan Meteorological Department (PMD)** website.\n*   Checking the weather app on your smartphone.\n\nThese sources will provide you with the current temperature, humidity, wind speed, and any weather alerts for the day.'

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)
pp(response['messages'])

[
    HumanMessage(content='How up to date is your training knowledge?', additional_kwargs={}, response_metadata={}, id='a02b437d-eb94-46ad-b30b-e5a308ea4dd0'),
    AIMessage(content="My training data cutoff date is **2026**. This means my knowledge and understanding of events, facts, and developments are current up to that point. For information beyond this date, I may not have accurate or complete details. If you have specific questions, I'll do my best to provide relevant and up-to-date answers based on my training!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 21, 'total_tokens': 93, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen35-122b-a10b', 'system_fingerprint': None, 'id': 'chatcmpl_75207e86-b1e6-4e12-a891-ab0911b9540d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eedcc-ee63-7bd2-a902-00009ffad7c1-0', tool_calls=[], invalid_too

## Add web search tool

In [5]:
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_KEY"))

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

resp = web_search.invoke("Who is the current mayor of Lahore?")
pp(resp)

{
    'answer': None,
    'follow_up_questions': None,
    'images': [],
    'query': 'Who is the current mayor of Lahore?',
    'request_id': '49ef88a9-ef27-4f2d-b351-551f5f8b8044',
    'response_time': 0.89,
    'results': [
        {
            'content': "The ruling party-backed Col (retd) Mubashir Javaid was elected unopposed as Lahore's mayor on Saturday after the Election Commission of Pakistan (ECP) rejected",
            'raw_content': None,
            'score': 0.8839694,
            'title': "Elected unopposed: Lahore's new mayor vows to serve the people",
            'url': 'https://tribune.com.pk/story/1259677/elected-unopposed-lahores-new-mayor-vows-serve-people',
        },
        {
            'content': 'Mayors of Lahore ; Mian Amir Mehmood, 2005, 2009 ; Administrator system implemented 2010–2016 ; Mubashar Javed, 2016, 2021 ; Administrator system implemented',
            'raw_content': None,
            'score': 0.8585411,
            'title': 'Mayor of Lahore - Wi

In [23]:
agent = create_agent(
    llm,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of Lahore on June 1, 2026?")

response = agent.invoke(
    {"messages": [question]}
)

In [15]:
pp(response['messages'])

[
    HumanMessage(content='Who is the current mayor of Lahore on June 1, 2026?', additional_kwargs={}, response_metadata={}, id='3d532f90-8afe-41e9-866c-8e983af78983'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 290, 'total_tokens': 385, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen35-122b-a10b', 'system_fingerprint': None, 'id': 'chatcmpl_fd1814ba-eb82-4b56-9d78-db9c597c24f1', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eedd2-3298-7fc2-a636-0eb6f8be09b4-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of Lahore 2026'}, 'id': 'call_173b6e43-9924-4190-807a-196b2d847dcd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 290, 'output_tokens': 95, 'total_tokens': 385, 'input_token_details': {}, 'output_token_details': {}}),
    ToolMessage(content='{"que

In [16]:
pp(response['messages'][-1].content)

[
    {
        'summary': [
            {
                'text': "Okay, the user is asking for the current mayor of Lahore on June 1, 2026. Let me start by recalling the last known mayor. From previous searches, I remember that Mubashir Javed was elected in 2016 and his term ended in 2021. After that, there was an administrator system. But wait, the user is asking about 2026, which is in the future. \n\nFirst, I need to check if there have been any elections after 2021. The Punjab Local Government Act 2013 sets a four-year term. So the next election should be around 2025. But the user's date is June 1, 2026. If the election was held in 2025, the new mayor would take office by then. However, if the election hasn't happened yet, the position might still be under an administrator.\n\nLooking at the search results, there's mention of a 2026 election schedule. Some results indicate that the ECP might have announced elections for 2026. But the exact date isn't clear. If the election is sch

In [17]:
pp(response["messages"][1].tool_calls)

[
    {
        'args': {'query': 'current mayor of Lahore 2026'},
        'id': 'call_173b6e43-9924-4190-807a-196b2d847dcd',
        'name': 'web_search',
        'type': 'tool_call',
    },
]


In [18]:
    question = HumanMessage(content="How is the weather today in Lahore?")

    response = agent.invoke(
        {"messages": [question]}
    )
    pp(response['messages'][-1].content)


'Based on the latest data, the weather in **Lahore** today is:\n\n*   **Condition:** Partly cloudy\n*   **Temperature:** Approximately **30°C (86°F)**\n*   **Feels Like:** Around **28.5°C (83°F)**\n*   **Humidity:** 59%\n*   **Wind:** Light breeze from the West at about 6.5 km/h (4 mph)\n*   **Chance of Rain:** Very low (1%)\n\nIt is currently daytime with moderate visibility. Please note that weather conditions can change rapidly, so checking a local forecast app for real-time updates is always a good idea.'


In [19]:
response['messages'][-1].content

'Based on the latest data, the weather in **Lahore** today is:\n\n*   **Condition:** Partly cloudy\n*   **Temperature:** Approximately **30°C (86°F)**\n*   **Feels Like:** Around **28.5°C (83°F)**\n*   **Humidity:** 59%\n*   **Wind:** Light breeze from the West at about 6.5 km/h (4 mph)\n*   **Chance of Rain:** Very low (1%)\n\nIt is currently daytime with moderate visibility. Please note that weather conditions can change rapidly, so checking a local forecast app for real-time updates is always a good idea.'